# PCA Anomaly Detection for TLS Profiling

This notebook evaluates one-class PCA baselines on extracted TLS traffic features. The dataset paths are parameterized so the same experiment can be run on one or more parquet partitions, and the target categories are controlled through `category_labels`. For each target label, the detector is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic using reconstruction error as the anomaly score.


In [1]:
import sys
from pathlib import Path
sys.path.append('../../src')

### PARAMETERS:

In [2]:
train_features_path = ["../data-ext/malware_train.parquet", "../data-ext/apps_train.parquet"]
val_features_path = ["../data-ext/malware_val.parquet", "../data-ext/apps_val.parquet"]
test_features_path = ["../data-ext/malware_test.parquet", "../data-ext/apps_test.parquet"]
train_labels_path = ["../data-ext/malware_train_labels.parquet", "../data-ext/apps_train_labels.parquet"]
val_labels_path = ["../data-ext/malware_val_labels.parquet", "../data-ext/apps_val_labels.parquet"]
test_labels_path = ["../data-ext/malware_test_labels.parquet", "../data-ext/apps_test_labels.parquet"]

category_labels = ["system", "malware", "application"]


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The experiments below sweep over every non-empty combination of these groups.


In [3]:
feature_groups = {
    "FLOW": ["bs", "ps", "br", "pr", "td"],
    "CTLS": ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"],
    "STLS": ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"],
    "REC": ["tls.rec"],
}

FLOW = feature_groups["FLOW"]
CTLS = feature_groups["CTLS"]
STLS = feature_groups["STLS"]
REC = feature_groups["REC"]

In [4]:
import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

def read_parquet_files(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    return pd.concat([pd.read_parquet(Path(path)) for path in paths], ignore_index=True)

print("Loading extracted features from parameterized parquet paths.")
df_train = read_parquet_files(train_features_path)
df_val = read_parquet_files(val_features_path)
df_test = read_parquet_files(test_features_path)
df_train_label = read_parquet_files(train_labels_path)
df_val_label = read_parquet_files(val_labels_path)
df_test_label = read_parquet_files(test_labels_path)

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


Loading extracted features from parameterized parquet paths.


Built preprocessor from df_train.


In [5]:
from tls_profiling.baselines.model_pca import PCADetector, Config
import numpy as np

def train_detector(train: np.ndarray) -> PCADetector:
    detector = PCADetector(Config())
    detector.fit(train)
    return detector

from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"unknown class\(es\) .* will be ignored",
    category=UserWarning,
    module=r"sklearn\.preprocessing\._label",
)

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float("inf")

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def evaluate(feature_set, normal_label):
    # x_train: only normal traffic
    x_train = df_train.loc[df_train_label["category"] == normal_label].reset_index(drop=True)
    # x_val: mixed traffic used to tune the F1 threshold
    x_val = df_val
    y_val = (df_val_label["category"] != normal_label).astype(int)
    # y_test: 1 = anomaly, 0 = normal
    y_test = (df_test_label["category"] != normal_label).astype(int)
    x_test = df_test

    # prepare datasets
    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    # create detector on TRAIN
    detector = train_detector(x_train_transformed)

    # choose the F1 threshold on the mixed validation split
    val_scores = detector.score(x_val_transformed)
    threshold = choose_f1_threshold(y_val, val_scores)

    # evaluate on TEST
    anomaly_score = detector.score(x_test_transformed)

    return y_test, anomaly_score, threshold

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    """
    Write the intermediate data to CSV file.
    """
    feature_set_name = "_".join(feature_set)
    class_label_name = normal_label

    output_path = f"tmp/pca_{class_label_name}_{feature_set_name}.csv"
    pd.DataFrame({
        "y_test": y_test,
        "y_pred": y_pred,
        "anomaly_score": anomaly_score,
    }).to_csv(output_path, index=False)

def compute_scores(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)
    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return {"auc_score": auc, "ap_score": ap, "f1_score": f1}

def plot_curves(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="PCA PR-AUC",
        ax=axes[0],
    )
    axes[0].set_title(f"{normal_label} Precision-Recall")

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="PCA AUC-ROC",
        ax=axes[1],
    )
    axes[1].set_title(f"{normal_label} ROC Curve")

    plt.tight_layout()
    plt.show()


## Evaluation

For each label listed in `category_labels`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses three disjoint splits:

- `train`: only samples from the selected normal class are used to fit the detector
- `val`: a mixed split is used to choose the anomaly-score threshold that maximizes F1
- `test`: the final metrics are reported using the threshold selected on validation

Each row in the result table corresponds to one target class and one feature-group combination.


In [6]:
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)

        for label in category_labels:
            display(f"Scoring {label}@{selected_features}...")
            scores = compute_scores(selected_features, label)

            rows.append({
                "FeatureSet": feature_set_name,
                "ClassLabel": label,
                "AucScore": scores["auc_score"],
                "ApScore": scores["ap_score"],
                "F1Score": scores["f1_score"],
            })

results_df = pd.DataFrame(rows, columns=["FeatureSet", "ClassLabel", "AucScore", "ApScore", "F1Score"])
display(results_df)


"Scoring system@['bs', 'ps', 'br', 'pr', 'td']..."

"Scoring malware@['bs', 'ps', 'br', 'pr', 'td']..."

"Scoring application@['bs', 'ps', 'br', 'pr', 'td']..."

"Scoring system@['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv']..."

"Scoring malware@['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv']..."

...

"Scoring application@['tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

"Scoring system@['bs', 'ps', 'br', 'pr', 'td', 'tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

"Scoring malware@['bs', 'ps', 'br', 'pr', 'td', 'tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

"Scoring application@['bs', 'ps', 'br', 'pr', 'td', 'tls.cver', 'tls.ccs', 'tls.cext', 'tls.csg', 'tls.alpn', 'tls.csv', 'tls.sver', 'tls.scs', 'tls.sext', 'tls.ssv', 'tls.rec']..."

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
0,FLOW,system,0.829213,0.857559,0.840516
1,FLOW,malware,0.239267,0.426910,0.702706
2,FLOW,application,0.245263,0.978275,0.994414
3,CTLS,system,0.921250,0.939628,0.900189
4,CTLS,malware,0.778384,0.806384,0.736670
5,CTLS,application,0.727748,0.993407,0.994414
6,STLS,system,0.805692,0.871335,0.882181
7,STLS,malware,0.859449,0.836567,0.738667
8,STLS,application,0.732672,0.994657,0.994414
9,REC,system,0.853550,0.894275,0.856105


## System Profile

The table below reports the PCA baseline for the `system` profile across all feature-group combinations. Sort by `F1Score`, `ApScore`, or `AucScore` to compare how reconstruction-error-based detection behaves relative to the other baselines.


In [7]:
system_df = results_df[results_df["ClassLabel"] == "system"].sort_values("F1Score", ascending=False)
display(system_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
27,"STLS, REC",system,0.937886,0.935134,0.944930
36,"FLOW, STLS, REC",system,0.937895,0.934459,0.944921
42,"FLOW, CTLS, STLS, REC",system,0.952663,0.967438,0.927080
39,"CTLS, STLS, REC",system,0.952640,0.967669,0.927018
30,"FLOW, CTLS, STLS",system,0.949313,0.965471,0.925437
12,"FLOW, CTLS",system,0.940311,0.963152,0.924034
33,"FLOW, CTLS, REC",system,0.937069,0.958186,0.920560
24,"CTLS, REC",system,0.936728,0.958200,0.913853
3,CTLS,system,0.921250,0.939628,0.900189
21,"CTLS, STLS",system,0.947775,0.963780,0.882582


## Malware Profile

This section isolates the PCA results for the `malware` profile. Comparing these rows against the `system` and `application` views highlights whether PCA benefits more from compact shared structure in malware traffic or struggles with intra-class diversity.


In [8]:
malware_df = results_df[results_df["ClassLabel"] == "malware"].sort_values("F1Score", ascending=False)
display(malware_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
40,"CTLS, STLS, REC",malware,0.820349,0.743746,0.846508
43,"FLOW, CTLS, STLS, REC",malware,0.820406,0.742686,0.845997
22,"CTLS, STLS",malware,0.818377,0.768881,0.838977
31,"FLOW, CTLS, STLS",malware,0.818944,0.742159,0.838905
28,"STLS, REC",malware,0.875780,0.851869,0.745794
37,"FLOW, STLS, REC",malware,0.876139,0.851456,0.745704
16,"FLOW, STLS",malware,0.865965,0.820550,0.742968
13,"FLOW, CTLS",malware,0.780253,0.715683,0.738725
7,STLS,malware,0.859449,0.836567,0.738667
4,CTLS,malware,0.778384,0.806384,0.736670


## Application Profile

This section isolates the PCA results for the `application` profile. Comparing it with the `system` and `malware` views shows whether application traffic shares enough low-dimensional structure for reconstruction error to detect out-of-profile samples reliably.


In [9]:
application_df = results_df[results_df["ClassLabel"] == "application"].sort_values("F1Score", ascending=False)
display(application_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
2,FLOW,application,0.245263,0.978275,0.994414
5,CTLS,application,0.727748,0.993407,0.994414
8,STLS,application,0.732672,0.994657,0.994414
11,REC,application,0.745483,0.995957,0.994414
20,"FLOW, REC",application,0.729314,0.995625,0.994414
29,"STLS, REC",application,0.769868,0.995995,0.994414
23,"CTLS, STLS",application,0.771630,0.993722,0.994414
44,"FLOW, CTLS, STLS, REC",application,0.804185,0.994734,0.994414
38,"FLOW, STLS, REC",application,0.764603,0.995854,0.994414
41,"CTLS, STLS, REC",application,0.804768,0.995030,0.994414
